# Topic 15 — Feature Scaling
### Theory → tiny example → why it matters → sklearn scalers → experiment.

Many algorithms are sensitive to the *scale* of features. If one feature ranges 0-1 and another
ranges 0-100,000, algorithms based on **distance** (KNN, SVM, K-Means) or **gradient-based
optimization** (linear/logistic regression, neural nets) can behave badly — the large-scale
feature dominates purely because of its units, not because it's actually more important.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)

## 1. The problem, illustrated

Two features on very different scales: `age` (0-100) and `income` (0-200,000).
Euclidean distance (Topic 10) will be almost entirely determined by `income`, drowning out `age`.

In [ ]:
data = np.array([
    [25, 40000],
    [30, 42000],
    [70, 39000],
])
# person 0 vs person 1: age differs by 5, income differs by 2000
# person 0 vs person 2: age differs by 45, income differs by 1000

dist_0_1 = np.linalg.norm(data[0] - data[1])
dist_0_2 = np.linalg.norm(data[0] - data[2])
print("raw distance 0 to 1:", dist_0_1)
print("raw distance 0 to 2:", dist_0_2)
# Distance to person 2 (45 years apart!) looks SMALLER than to person 1 (only 5 years apart) --
# purely because income differences (thousands) dwarf age differences (tens) in raw units.

## 2. Standardization (Z-score scaling)

Transforms each feature to have **mean 0 and standard deviation 1**:

```text
x_scaled = (x - mean) / std
```

The most common default choice. Doesn't force data into a fixed range, but centers and rescales it.

In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
print("standardized data:\n", data_scaled)
print("mean per column (should be ~0):", data_scaled.mean(axis=0))
print("std per column (should be ~1):", data_scaled.std(axis=0))

dist_0_1_scaled = np.linalg.norm(data_scaled[0] - data_scaled[1])
dist_0_2_scaled = np.linalg.norm(data_scaled[0] - data_scaled[2])
print("\nscaled distance 0 to 1:", dist_0_1_scaled)
print("scaled distance 0 to 2:", dist_0_2_scaled)
# Now the 45-year age gap correctly shows up as the bigger distance.

## 3. Min-max scaling (normalization)

Rescales each feature into a fixed range, usually [0, 1]:

```text
x_scaled = (x - min) / (max - min)
```

Useful when you want values bounded in a known range (e.g. for some neural net input layers),
but sensitive to outliers (a single extreme value compresses everything else).

In [ ]:
minmax = MinMaxScaler()
data_minmax = minmax.fit_transform(data)
print("min-max scaled data:\n", data_minmax)
print("min per column:", data_minmax.min(axis=0), " max per column:", data_minmax.max(axis=0))

## 4. RobustScaler — resistant to outliers

Uses the **median** and **IQR (interquartile range)** instead of mean/std, so extreme outliers don't
skew the scaling as much.

In [ ]:
data_with_outlier = np.array([[20], [22], [21], [23], [500]])   # one huge outlier

standard_result = StandardScaler().fit_transform(data_with_outlier)
robust_result = RobustScaler().fit_transform(data_with_outlier)

print("original:       ", data_with_outlier.flatten())
print("StandardScaler: ", np.round(standard_result.flatten(), 2))
print("RobustScaler:   ", np.round(robust_result.flatten(), 2))
# StandardScaler squashes the 4 normal points very close together because the outlier inflates
# the std. RobustScaler keeps them more spread out and meaningfully separated.

## 5. Effect on a real algorithm: KNN with vs without scaling

Reusing KNN (Topic 10), which is purely distance-based -- the algorithm most directly harmed
by unscaled features.

In [ ]:
X, y = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0,
                            n_clusters_per_class=1, class_sep=1.2, random_state=42)

# Artificially blow up the scale of feature 1 to simulate mismatched units
X_unscaled = X.copy()
X_unscaled[:, 1] *= 1000

X_train, X_test, y_train, y_test = train_test_split(X_unscaled, y, test_size=0.3, random_state=42)

# Without scaling
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn_raw.predict(X_test))

# With scaling (fit scaler on train ONLY -- Topic 5, avoid data leakage)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
knn_scaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))

print("KNN accuracy WITHOUT scaling:", acc_raw)
print("KNN accuracy WITH scaling:   ", acc_scaled)
# Scaling should noticeably help here since feature 1 was artificially blown out of proportion.

## 6. Which algorithms actually need scaling?

| Needs scaling | Doesn't need scaling |
|---|---|
| KNN | Decision Trees |
| SVM | Random Forest |
| Logistic/Linear Regression (for faster gradient descent convergence) | Naive Bayes (works on counts/frequencies directly) |
| Neural Networks | |
| K-Means, PCA | |

Tree-based models split on raw feature values one feature at a time, so relative scale between
*different* features doesn't affect their splits.

In [ ]:
# --- Try it yourself ---
# 1. Repeat the KNN experiment above but scale feature 1 by 100000 instead of 1000 -- does the gap widen?
# 2. Retrain a DecisionTreeClassifier (Topic 12) on X_unscaled vs a scaled version --
#    confirm accuracy stays basically the same either way (trees are scale-invariant).
# 3. Apply MinMaxScaler to data_with_outlier from part 4 -- how badly does the outlier compress
#    the other 4 points into a tiny range near 0?
# 4. In your own words: why must you ALWAYS fit a scaler on train data only, then .transform()
#    (not .fit_transform()) the test data? (Reconnect to Topic 5's data leakage discussion.)

---
### Next up: **Topic 16 — Feature Engineering**.

Say "next" when you're ready.